# <center>EMPLOYEE LOAD</center>
```sql
SELECT
    e.BusinessEntityID, p.Title, p.FirstName, p.MiddleName, p.LastName, p.Suffix, e.JobTitle, pp.PhoneNumber
    ,pnt.Name AS PhoneNumberType, ea.EmailAddress, p.EmailPromotion, a.AddressLine1, a.AddressLine2, a.City, 
    sp.Name AS StateProvinceName, a.PostalCode, cr.Name AS CountryRegionName, p.AdditionalContactInfo
FROM HumanResources.Employee e
	INNER JOIN Person.Person p
	ON p.BusinessEntityID = e.BusinessEntityID
    INNER JOIN Person.BusinessEntityAddress bea
    ON bea.BusinessEntityID = e.BusinessEntityID
    INNER JOIN Person.Address a
    ON a.AddressID = bea.AddressID
    INNER JOIN Person.StateProvince sp
    ON sp.StateProvinceID = a.StateProvinceID
    INNER JOIN Person.CountryRegion cr
    ON cr.CountryRegionCode = sp.CountryRegionCode
    LEFT OUTER JOIN Person.PersonPhone pp
    ON pp.BusinessEntityID = p.BusinessEntityID
    LEFT OUTER JOIN Person.PhoneNumberType pnt
    ON pp.PhoneNumberTypeID = pnt.PhoneNumberTypeID
    LEFT OUTER JOIN Person.EmailAddress ea
    ON p.BusinessEntityID = ea.BusinessEntityID;
```


In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_data_warehouse"
CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.employee_Staging"
wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog("local")

spark

```mermaid
flowchart
direction LR

Employee_BusinessEntityID -- INSERT --> staging_BusinessEntityID
Person_Title -- INSERT --> staging_Title
Person_FirstName -- INSERT --> staging_FirstName
Person_MiddleName -- INSERT --> staging_MiddleName
Person_LastName -- INSERT --> staging_LastName
Person_Suffix -- INSERT --> staging_Suffix
Employee_JobTitle -- INSERT --> staging_JobTitle
PersonPhone_PhoneNumber -- INSERT --> staging_PhoneNumber
PhoneNumberType_Name -- INSERT --> staging_PhoneNumberType
EmailAddress_EmailAddress -- INSERT --> staging_EmailAddress
Person_EmailPromotion -- INSERT --> staging_EmailPromotion
Address_AddressLine1 -- INSERT --> staging_AddressLine1
Address_AddressLine2 -- INSERT --> staging_AddressLine2
Address_City -- INSERT --> staging_City
StateProvince_Name -- INSERT --> staging_StateProvinceName
Address_PostalCode -- INSERT --> staging_PostalCode
CountryRegion_Name -- INSERT --> staging_CountryRegionName
Person_AdditionalContactInfo -- INSERT --> staging_AdditionalContactInfo

staging_BusinessEntityID ins1@-- UPSERT -->dimension_BusinessEntityID
staging_Title ins2@-- UPSERT -->dimension_Title
staging_FirstName ins3@-- UPSERT -->dimension_FirstName
staging_MiddleName ins4@-- UPSERT -->dimension_MiddleName
staging_LastName ins5@-- UPSERT -->dimension_LastName
staging_Suffix ins6@-- UPSERT -->dimension_Suffix
staging_JobTitle ins7@-- UPSERT -->dimension_JobTitle
staging_PhoneNumber ins8@-- UPSERT -->dimension_PhoneNumber
staging_PhoneNumberType ins9@-- UPSERT -->dimension_PhoneNumberType
staging_EmailAddress ins10@-- UPSERT -->dimension_EmailAddress
staging_EmailPromotion in11@-- UPSERT -->dimension_EmailPromotion
staging_AddressLine1 ins12@-- UPSERT -->dimension_AddressLine1
staging_AddressLine2 ins13@-- UPSERT -->dimension_AddressLine2
staging_City ins14@-- UPSERT -->dimension_City
staging_StateProvinceName ins15@-- UPSERT -->dimension_StateProvinceName
staging_PostalCode ins16@-- UPSERT -->dimension_PostalCode
staging_CountryRegionName ins17@-- UPSERT -->dimension_CountryRegionName
staging_AdditionalContactInfo ins18@-- UPSERT -->dimension_AdditionalContactInfo

ins1@{animation: fast}
ins2@{animation: fast}
ins3@{animation: fast}
ins4@{animation: fast}
ins5@{animation: fast}
ins6@{animation: fast}
ins7@{animation: fast}
ins8@{animation: fast}
ins9@{animation: fast}
ins10@{animation: fast}
in11@{animation: fast}
ins12@{animation: fast}
ins13@{animation: fast}
ins14@{animation: fast}
ins15@{animation: fast}
ins16@{animation: fast}
ins17@{animation: fast}
ins18@{animation: fast}


Person_BusinessEntityID j1@ o-.Join.-o Employee_BusinessEntityID
BusinessEntityAddress_BusinessEntityID j2@ o-.Join.-o Employee_BusinessEntityID
Address_AddressID j3@ o-.Join.-o BusinessEntityAddress_AddressID
StateProvince_StateProvinceID j4@ o-.Join.-o Address_StateProvinceID
CountryRegion_CountryRegionCode j5@ o-.Join.-o StateProvince_CountryRegionCode
PersonPhone_BusinessEntityID j6@ o-.Join.-o Person_BusinessEntityID
PersonPhone_PhoneNumberTypeID j7@ o-.Join.-o PhoneNumberType_PhoneNumberTypeID
Person_BusinessEntityID j8@ o-.Join.-o EmailAddress_BusinessEntityID
j1@{animation: slow}
j2@{animation: slow}
j3@{animation: slow}
j4@{animation: slow}
j5@{animation: slow}
j6@{animation: slow}
j7@{animation: slow}
j8@{animation: slow}



subgraph source
    subgraph HumanResources.Employee
        direction TB
        Employee_JobTitle[JobTitle]
        Employee_BusinessEntityID[BusinessEntityID]
    end
    subgraph Person.Person
        direction TB
        Person_BusinessEntityID[BusinessEntityID] 
        Person_Title[Title]
        Person_FirstName[FirstName]
        Person_MiddleName[MiddleName]
        Person_LastName[LastName]
        Person_Suffix[Suffix]
        Person_EmailPromotion[EmailPromotion]
        Person_AdditionalContactInfo[AdditionalContactInfo]
    end
    subgraph Person.BusinessEntityAddress
        direction TB
        BusinessEntityAddress_BusinessEntityID[BusinessEntityID] 
        BusinessEntityAddress_AddressID[AddressID]
    end
    subgraph Person.PhoneNumberType
        direction TB
        PhoneNumberType_Name[Name]
        PhoneNumberType_PhoneNumberTypeID[PhoneNumberTypeID]
    end
    subgraph Person.Address
        direction TB
        Address_AddressID[AddressID]
        Address_StateProvinceID[StateProvinceID]
        Address_AddressLine1[AddressLine1]
        Address_AddressLine2[AddressLine2]
        Address_City[City]
        Address_PostalCode[PostalCode]
    end
    subgraph Person.StateProvince
        StateProvince_Name[Name]
        StateProvince_StateProvinceID[StateProvinceID]
        StateProvince_CountryRegionCode[CountryRegionCode]
    end
    subgraph Person.CountryRegion
        CountryRegion_Name[Name]
        CountryRegion_CountryRegionCode[CountryRegionCode]
    end
    subgraph Person.PersonPhone
        PersonPhone_PhoneNumber[PhoneNumber]
        PersonPhone_BusinessEntityID[BusinessEntityID]
        PersonPhone_PhoneNumberTypeID[PhoneNumberTypeID]
    end
    subgraph Person.PhoneNumberType
        PhoneNumberType_Name[PhoneNumberType]
        PhoneNumberType_PhoneNumberTypeID[PhoneNumberTypeID]
    end
    subgraph Person.EmailAddress
        direction TB
        EmailAddress_EmailAddress[EmailAddress]
        EmailAddress_BusinessEntityID[BusinessEntityID]
        EmailAddress_BusinessEntityID[BusinessEntityID]
    end
end
subgraph destination
    subgraph staging_employees        
        staging_BusinessEntityID[BusinessEntityID]
        staging_Title[Title]
        staging_FirstName[FirstName]
        staging_MiddleName[MiddleName]
        staging_LastName[LastName]
        staging_Suffix[Suffix]
        staging_JobTitle[JobTitle]
        staging_PhoneNumber[PhoneNumber]
        staging_PhoneNumberType[PhoneNumberType]
        staging_EmailAddress[EmailAddress]
        staging_EmailPromotion[EmailPromotion]
        staging_AddressLine1[AddressLine1]
        staging_AddressLine2[AddressLine2]
        staging_City[City]
        staging_StateProvinceName[StateProvinceName]
        staging_PostalCode[PostalCode]
        staging_CountryRegionName[CountryRegionName]
        staging_AdditionalContactInfo[AdditionalContactInfo]
    end
    subgraph dimension_employees
        dimension_BusinessEntityID[BusinessEntityID]
        dimension_Title[Title]
        dimension_FirstName[FirstName]
        dimension_MiddleName[MiddleName]
        dimension_LastName[LastName]
        dimension_Suffix[Suffix]
        dimension_JobTitle[JobTitle]
        dimension_PhoneNumber[PhoneNumber]
        dimension_PhoneNumberType[PhoneNumberType]
        dimension_EmailAddress[EmailAddress]
        dimension_EmailPromotion[EmailPromotion]
        dimension_AddressLine1[AddressLine1]
        dimension_AddressLine2[AddressLine2]
        dimension_City[City]
        dimension_StateProvinceName[StateProvinceName]
        dimension_PostalCode[PostalCode]
        dimension_CountryRegionName[CountryRegionName]
        dimension_AdditionalContactInfo[AdditionalContactInfo]
        dimension_recordHash[RecordHash]
        dimension_ValidFrom[ValidFrom]
        dimension_ValidTo[ValidTo]
        dimension_isActive[isActive]
    end
end
```

In [ ]:
df_HumanResources_Employee = spark.table("HumanResources.Employee").alias("e")
df_Person_Person = spark.table("Person.Person").alias("p")
df_Person_BusinessEntityAddress = spark.table("Person.BusinessEntityAddress").alias("bea")
df_Person_Address = spark.table("Person.Address").alias("a")
df_Person_StateProvince = spark.table("Person.StateProvince").alias("sp")
df_Person_CountryRegion = spark.table("Person.CountryRegion").alias("cr")
df_Person_PersonPhone = spark.table("Person.PersonPhone").alias("pp")
df_Person_PhoneNumberType = spark.table("Person.PhoneNumberType").alias("pnt")
df_Person_EmailAddress = spark.table("Person.EmailAddress").alias("ea")

In [ ]:
joined = (
    df_HumanResources_Employee
    .join(df_Person_Person, df_HumanResources_Employee.BusinessEntityID == df_Person_Person.BusinessEntityID, "inner")
    .join(df_Person_BusinessEntityAddress, df_Person_Person.BusinessEntityID == df_Person_BusinessEntityAddress.BusinessEntityID, "inner")
    .join(df_Person_Address, df_Person_BusinessEntityAddress.AddressID == df_Person_Address.AddressID, "inner")
    .join(df_Person_StateProvince, df_Person_Address.StateProvinceID == df_Person_StateProvince.StateProvinceID, "inner")
    .join(df_Person_CountryRegion, df_Person_StateProvince.CountryRegionCode == df_Person_CountryRegion.CountryRegionCode, "inner")
    .join(df_Person_PersonPhone, df_Person_Person.BusinessEntityID == df_Person_PersonPhone.BusinessEntityID, "left")
    .join(df_Person_PhoneNumberType, df_Person_PersonPhone.PhoneNumberTypeID == df_Person_PhoneNumberType.PhoneNumberTypeID, "left")
    .join(df_Person_EmailAddress, df_Person_Person.BusinessEntityID == df_Person_EmailAddress.BusinessEntityID, "left")
)\
    .select(
    sf.coalesce(sf.col("e.BusinessEntityID").cast("int"),sf.lit(None)).alias("BusinessEntityID"),
    sf.coalesce(sf.col("p.Title"),sf.lit(None)).alias("Title"),
    sf.coalesce(sf.col("p.FirstName"),sf.lit(None)).alias("FirstName"),
    sf.coalesce(sf.col("p.MiddleName"),sf.lit(None)).alias("MiddleName"),
    sf.coalesce(sf.col("p.LastName"),sf.lit(None)).alias("LastName"),
    sf.coalesce(sf.col("p.Suffix"),sf.lit(None)).alias("Suffix"),
    sf.coalesce(sf.col("e.JobTitle"),sf.lit(None)).alias("JobTitle"),
    sf.coalesce(sf.col("pp.PhoneNumber"),sf.lit(None)).alias("PhoneNumber"),
    sf.coalesce(sf.col("pnt.Name"),sf.lit(None)).alias("PhoneNumberType"),
    sf.coalesce(sf.col("ea.EmailAddress"),sf.lit(None)).alias("EmailAddress"),
    sf.coalesce(sf.col("p.EmailPromotion").cast("int"),sf.lit(None)).alias("EmailPromotion"),
    sf.coalesce(sf.col("a.AddressLine1"),sf.lit(None)).alias("AddressLine1"),
    sf.coalesce(sf.col("a.AddressLine2"),sf.lit(None)).alias("AddressLine2"),
    sf.coalesce(sf.col("a.City"),sf.lit(None)).alias("City"),
    sf.coalesce(sf.col("sp.Name"),sf.lit(None)).alias("StateProvinceName"),
    sf.coalesce(sf.col("a.PostalCode"),sf.lit(None)).alias("PostalCode"),
    sf.coalesce(sf.col("cr.Name"),sf.lit(None)).alias("CountryRegionName"),
    sf.coalesce(sf.col("p.AdditionalContactInfo"),sf.lit(None)).alias("AdditionalContactInfo"),
    )
df_employeeStaging = joined.dropDuplicates(["BusinessEntityID"])
df_employeeStaging.show(10, False)
df_employeeStaging.printSchema()

df_employeeStaging.writeTo(staging_table_name) \
    .partitionedBy("CountryRegionName", "StateProvinceName", "City") \
    .using("iceberg") \
    .createOrReplace()

In [ ]:
# # spark.sql("SHOW TBLPROPERTIES " + staging_table_name).show(truncate=False)
# spark.sql("DESCRIBE TABLE EXTENDED " + staging_table_name).show(30, truncate=False)
# spark.read.table(staging_table_name + ".partitions").show(truncate=False)

In [ ]:
df_source = spark.read.table(staging_table_name)
df_source = df_source\
    .withColumn("recordHash", sf.md5(sf.concat_ws("||",
        sf.col("BusinessEntityID").cast("string"),
        sf.col("Title"),
        sf.col("FirstName"),
        sf.col("MiddleName"),
        sf.col("LastName"),
        sf.col("Suffix"),
        sf.col("JobTitle"),
        sf.col("PhoneNumber"),
        sf.col("PhoneNumberType"),
        sf.col("EmailAddress"),
        sf.col("EmailPromotion").cast("string"),
        sf.col("AddressLine1"),
        sf.col("AddressLine2"),
        sf.col("City"),
        sf.col("StateProvinceName"),
        sf.col("PostalCode"),
        sf.col("CountryRegionName"),
        sf.col("AdditionalContactInfo")
    )))\
    .withColumn("validFrom", sf.lit(datetime.now()))\
    .withColumn("validTill", sf.lit(None).cast(sdt.TimestampType()))\
    .withColumn("isActive", sf.lit(1).cast(sdt.IntegerType()))

df_source.show(10, False)

In [ ]:
spark.catalog.setCurrentCatalog("staging")
spark.stop()